# MT-SICS Backend Hardware Validation

Run this notebook connected to a physical Mettler Toledo scale to validate
every implemented backend method. Each cell tests one method and logs the
raw MT-SICS command/response.

**Requirements:**
- Physical Mettler Toledo scale connected via USB-to-serial
- Scale powered on and warmed up (60 min)
- Weighing pan empty at start

---
## 1. Setup and Logging

In [1]:
import logging
from datetime import datetime

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO
from pylabrobot.scales import Scale
from pylabrobot.scales.mettler_toledo import MettlerToledoWXS205SDUBackend

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = f"./logs/hardware_validation/{timestamp}_validation.log"

pylabrobot.verbose(True, level=LOG_LEVEL_IO)
pylabrobot.setup_logger(log_dir=log_file, level=LOG_LEVEL_IO)

plr_logger = logging.getLogger("pylabrobot")
plr_logger.info("=== Hardware validation started ===")

2026-03-30 11:28:29,103 - pylabrobot - INFO - === Hardware validation started ===


In [3]:
# Update port for your system
backend = MettlerToledoWXS205SDUBackend(port="/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0")
scale = Scale(name="validation_scale", backend=backend, size_x=0, size_y=0, size_z=0)

await scale.setup()

2026-03-30 11:29:07,453 - pylabrobot.io.serial - INFO - Using explicitly provided port: /dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0 (for VID=1027, PID=24577)
2026-03-30 11:29:07,456 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 11:29:07,457 - pylabrobot - IO - [MT Scale] Sent command: @
2026-03-30 11:29:07,459 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 11:29:07,520 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 11:29:07,522 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'
2026-03-30 11:29:07,523 - pylabrobot - IO - [MT Scale] Sent command: I1
2026-03-30 11:29:07,525 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I1\r\n'
2026-03-30 11:29:07,569 - pylabrobot.io.serial - IO - [/dev/serial

### Verify setup populated device identity

In [4]:
print(f"Device type:    {backend.device_type}")
print(f"Serial number:  {backend.serial_number}")
print(f"Capacity:       {backend.capacity} g")
print(f"MT-SICS levels: {sorted(backend._mt_sics_levels)}")

assert backend.device_type is not None
assert backend.serial_number is not None
assert backend.capacity > 0
assert 0 in backend._mt_sics_levels
print("PASS: setup populated all identity fields")

Device type:    WXS205SDU WXA-Bridge
Serial number:  B207696838
Capacity:       220.009 g
MT-SICS levels: [0, 1]
PASS: setup populated all identity fields


### I0 - Discover all implemented commands

This tells us exactly which MT-SICS commands this specific device supports,
resolving whether SC, C, D are truly unsupported or just miscategorized by level.

In [ ]:
from collections import defaultdict

# I0 is a multi-response command (B status for each command, A for the last)
responses = await backend.send_command("I0")

print(f"Total commands implemented: {len(responses)}")
print()

# Group by level
by_level = defaultdict(list)
for resp in responses:
    # Format: I0 B/A <level> <"command">
    if len(resp.data) >= 2:
        level = resp.data[0]
        cmd_name = resp.data[1]
        by_level[level].append(cmd_name)

for level in sorted(by_level.keys()):
    cmds = sorted(by_level[level])
    print(f"Level {level} ({len(cmds)} commands): {', '.join(cmds)}")

# Check which commands we use that might not be supported
our_commands = ["@", "C", "D", "DW", "I0", "I1", "I2", "I4", "I50",
                "M21", "M28", "S", "SC", "SI", "T", "TA", "TAC",
                "TI", "Z", "ZC", "ZI", "TC"]
all_device_cmds = set()
for cmds in by_level.values():
    all_device_cmds.update(cmds)

print()
print("Commands we use vs device support:")
for cmd in sorted(our_commands):
    status = "supported" if cmd in all_device_cmds else "NOT SUPPORTED"
    print(f"  {cmd}: {status}")

---
## 2. Level 0 Commands (Basic Set)

### @ - Cancel (reset to determined state)

In [5]:
serial_number = await backend.cancel()
print(f"Cancel returned serial number: {serial_number}")
assert serial_number == backend.serial_number
print("PASS: cancel returns correct serial number")

2026-03-30 11:29:22,637 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 11:29:22,638 - pylabrobot - IO - [MT Scale] Sent command: @
2026-03-30 11:29:22,639 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 11:29:22,693 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 11:29:22,694 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'


Cancel returned serial number: B207696838
PASS: cancel returns correct serial number


### I4 - Serial number

In [6]:
sn = await backend.request_serial_number()
print(f"Serial number: {sn}")
assert len(sn) > 0
print("PASS: serial number is non-empty")

2026-03-30 11:29:29,649 - pylabrobot - IO - [MT Scale] Sent command: I4
2026-03-30 11:29:29,651 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I4\r\n'
2026-03-30 11:29:29,694 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 11:29:29,696 - pylabrobot - IO - [MT Scale] Received response: b'I4 A "B207696838"\r\n'


Serial number: B207696838
PASS: serial number is non-empty


### I2 - Device type and capacity

In [7]:
device_type = await backend.request_device_type()
capacity = await backend.request_capacity()
print(f"Device type: {device_type}")
print(f"Capacity: {capacity} g")
assert len(device_type) > 0
assert capacity > 0
print("PASS: device type and capacity valid")

2026-03-30 11:29:32,001 - pylabrobot - IO - [MT Scale] Sent command: I2
2026-03-30 11:29:32,005 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I2\r\n'
2026-03-30 11:29:32,061 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'
2026-03-30 11:29:32,062 - pylabrobot - IO - [MT Scale] Received response: b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'
2026-03-30 11:29:32,064 - pylabrobot - IO - [MT Scale] Sent command: I2
2026-03-30 11:29:32,065 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I2\r\n'
2026-03-30 11:29:32,124 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'
2026-03-30 11:29:32,125 - pylabrobot - IO - [MT Scale] Received response: b'I2 A "WXS205SDU WXA-Bridge 220.00900 g"\r\n'


Device type: WXS205SDU WXA-Bridge
Capacity: 220.009 g
PASS: device type and capacity valid


### I1 - MT-SICS levels

In [8]:
levels = await backend._query_mt_sics_levels()
print(f"Supported levels: {sorted(levels)}")
assert 0 in levels
assert 1 in levels
print("PASS: levels 0 and 1 present (always required)")

2026-03-30 11:29:34,955 - pylabrobot - IO - [MT Scale] Sent command: I1
2026-03-30 11:29:34,959 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I1\r\n'
2026-03-30 11:29:35,002 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I1 A "01" "2.30" "2.22" "" ""\r\n'
2026-03-30 11:29:35,004 - pylabrobot - IO - [MT Scale] Received response: b'I1 A "01" "2.30" "2.22" "" ""\r\n'


Supported levels: [0, 1]
PASS: levels 0 and 1 present (always required)


### S - Stable weight (ensure pan is empty)

In [9]:
weight = await backend.read_stable_weight()
print(f"Stable weight (empty pan): {weight} g")
assert isinstance(weight, float)
print("PASS: stable weight returns float")

2026-03-30 11:29:40,982 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 11:29:40,987 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 11:29:41,078 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S   -0.00102 g\r\n'
2026-03-30 11:29:41,079 - pylabrobot - IO - [MT Scale] Received response: b'S S   -0.00102 g\r\n'


Stable weight (empty pan): -0.00102 g
PASS: stable weight returns float


### SI - Weight immediately

In [10]:
weight = await backend.read_weight_value_immediately()
print(f"Immediate weight: {weight} g")
assert isinstance(weight, float)
print("PASS: immediate weight returns float")

2026-03-30 11:29:44,419 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 11:29:44,422 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 11:29:44,466 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S   -0.00102 g\r\n'
2026-03-30 11:29:44,469 - pylabrobot - IO - [MT Scale] Received response: b'S S   -0.00102 g\r\n'


Immediate weight: -0.00102 g
PASS: immediate weight returns float


### Z - Zero (stable)

In [11]:
await scale.zero(timeout="stable")
weight = await backend.read_weight_value_immediately()
print(f"Weight after zero: {weight} g")
assert abs(weight) < 0.001, f"Expected ~0 after zero, got {weight}"
print("PASS: zero sets weight to ~0")

2026-03-30 11:29:47,415 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 11:29:47,419 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 11:29:47,647 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 11:29:47,648 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 11:29:47,648 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 11:29:47,650 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 11:29:47,680 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 11:29:47,682 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'


Weight after zero: 0.0 g
PASS: zero sets weight to ~0


### ZI - Zero immediately

In [12]:
await scale.zero(timeout=0)
weight = await backend.read_weight_value_immediately()
print(f"Weight after zero immediately: {weight} g")
assert abs(weight) < 0.01
print("PASS: zero immediately sets weight to ~0")

2026-03-30 11:29:51,308 - pylabrobot - IO - [MT Scale] Sent command: ZI
2026-03-30 11:29:51,312 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'ZI\r\n'
2026-03-30 11:29:51,341 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ZI S\r\n'
2026-03-30 11:29:51,341 - pylabrobot - IO - [MT Scale] Received response: b'ZI S\r\n'
2026-03-30 11:29:51,342 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 11:29:51,344 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 11:29:51,373 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 11:29:51,374 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'


Weight after zero immediately: 0.0 g
PASS: zero immediately sets weight to ~0


---
## 3. Level 1 Commands (Elementary)

**Place a known weight on the scale for tare tests.**

### T - Tare (stable)

In [13]:
input("Place a container on the scale and press Enter...")
await scale.zero(timeout="stable")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Tare weight: {tare} g")
assert tare > 0, f"Expected positive tare, got {tare}"
print("PASS: tare stores container weight")

Place a container on the scale and press Enter... 


2026-03-30 11:30:01,504 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 11:30:01,506 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 11:30:01,796 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 11:30:01,797 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 11:30:01,799 - pylabrobot - IO - [MT Scale] Sent command: T
2026-03-30 11:30:01,801 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 11:30:02,211 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 11:30:02,212 - pylabrobot - IO - [MT Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 11:30:02,213 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 11:30:02,214 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTD

Tare weight: 0.0 g


AssertionError: Expected positive tare, got 0.0

### TA - Request tare weight

In [14]:
tare = await scale.request_tare_weight()
print(f"Tare weight from memory: {tare} g")
assert isinstance(tare, float)
print("PASS: tare weight readable from memory")

2026-03-30 11:30:08,454 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 11:30:08,463 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 11:30:08,511 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 11:30:08,517 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Tare weight from memory: 0.0 g
PASS: tare weight readable from memory


### TAC - Clear tare

In [15]:
await backend.clear_tare()
tare_after_clear = await scale.request_tare_weight()
print(f"Tare after clear: {tare_after_clear} g")
assert abs(tare_after_clear) < 0.001, f"Expected ~0 after clear, got {tare_after_clear}"
print("PASS: clear tare resets tare to 0")

2026-03-30 11:30:10,949 - pylabrobot - IO - [MT Scale] Sent command: TAC
2026-03-30 11:30:10,950 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TAC\r\n'
2026-03-30 11:30:10,972 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TAC A\r\n'
2026-03-30 11:30:10,973 - pylabrobot - IO - [MT Scale] Received response: b'TAC A\r\n'
2026-03-30 11:30:10,973 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 11:30:10,975 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 11:30:11,054 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 11:30:11,056 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Tare after clear: 0.0 g
PASS: clear tare resets tare to 0


### SC - Dynamic weight (timed read)

In [16]:
weight = await backend.read_dynamic_weight(timeout=3.0)
print(f"Dynamic weight (3s timeout): {weight} g")
assert isinstance(weight, float)
print("PASS: dynamic weight returns float")

2026-03-30 11:30:12,434 - pylabrobot - IO - [MT Scale] Sent command: SC 3000
2026-03-30 11:30:12,438 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SC 3000\r\n'
2026-03-30 11:30:12,459 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 11:30:12,461 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


MettlerToledoError: Syntax error: The weigh module/balance has not recognized the received command or the command is not allowed

### C - Cancel all

In [17]:
await backend.cancel_all()
print("PASS: cancel_all completed without error")

2026-03-30 11:30:22,642 - pylabrobot - IO - [MT Scale] Sent command: C
2026-03-30 11:30:22,646 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'C\r\n'
2026-03-30 11:30:22,659 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 11:30:22,660 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


MettlerToledoError: Syntax error: The weigh module/balance has not recognized the received command or the command is not allowed

### D / DW - Display text

In [18]:
import asyncio

await backend.set_display_text("PLR TEST")
await asyncio.sleep(2)
await backend.set_weight_display()
print("PASS: display text set and restored (check terminal if connected)")

2026-03-30 11:30:25,587 - pylabrobot - IO - [MT Scale] Sent command: D "PLR TEST"
2026-03-30 11:30:25,590 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'D "PLR TEST"\r\n'
2026-03-30 11:30:25,617 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 11:30:25,618 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


MettlerToledoError: Syntax error: The weigh module/balance has not recognized the received command or the command is not allowed

### ZC / TC - Timed zero and tare

These are known to return ES (syntax error) on the WXS205SDU despite being in the spec.

In [19]:
from pylabrobot.scales.mettler_toledo.errors import MettlerToledoError

for cmd_name, cmd in [("ZC", "ZC 5000"), ("TC", "TC 5000")]:
  try:
    await backend.send_command(cmd)
    print(f"{cmd_name}: supported on this device")
  except MettlerToledoError as e:
    if "Syntax error" in str(e):
      print(f"{cmd_name}: not supported (ES) - expected for WXS205SDU")
    else:
      print(f"{cmd_name}: unexpected error: {e}")

2026-03-30 11:30:28,230 - pylabrobot - IO - [MT Scale] Sent command: ZC 5000
2026-03-30 11:30:28,233 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'ZC 5000\r\n'
2026-03-30 11:30:28,270 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 11:30:28,271 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'
2026-03-30 11:30:28,275 - pylabrobot - IO - [MT Scale] Sent command: TC 5000
2026-03-30 11:30:28,279 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TC 5000\r\n'
2026-03-30 11:30:28,302 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'ES\r\n'
2026-03-30 11:30:28,306 - pylabrobot - IO - [MT Scale] Received response: b'ES\r\n'


ZC: not supported (ES) - expected for WXS205SDU
TC: not supported (ES) - expected for WXS205SDU


---
## 4. Level 2 Commands (Extended)

### M21 - Set host unit to grams

In [20]:
if 2 in backend._mt_sics_levels:
  await backend.set_host_unit_grams()
  print("PASS: host unit set to grams")
else:
  print("SKIP: Level 2 not supported")

SKIP: Level 2 not supported


### I50 - Remaining weighing range (multi-response)

In [21]:
if 2 in backend._mt_sics_levels:
  remaining = await backend.request_remaining_weighing_range()
  print(f"Remaining weighing range: {remaining} g")
  assert remaining > 0
  assert remaining <= backend.capacity
  print("PASS: remaining range is positive and within capacity")
else:
  print("SKIP: Level 2 not supported")

SKIP: Level 2 not supported


### M28 - Temperature sensor

In [22]:
if 2 in backend._mt_sics_levels:
  temp = await backend.measure_temperature()
  print(f"Scale temperature: {temp} C")
  assert 5 < temp < 50, f"Temperature {temp} C outside reasonable range"
  print("PASS: temperature sensor returns reasonable value")
else:
  print("SKIP: Level 2 not supported")

SKIP: Level 2 not supported


---
## 5. Frontend Integration

Test the Scale frontend methods that delegate to the backend.

In [23]:
# Zero via frontend
await scale.zero(timeout="stable")
w = await scale.read_weight(timeout=0)
print(f"Frontend zero + read: {w} g")
assert abs(w) < 0.001

# Tare via frontend
input("Place a container on the scale and press Enter...")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Frontend tare weight: {tare} g")
assert tare > 0

# Read via frontend
w = await scale.read_weight(timeout="stable")
print(f"Frontend read (should be ~0 with container tared): {w} g")
assert abs(w) < 0.1

print("PASS: all frontend methods delegate correctly")

2026-03-30 11:30:52,063 - pylabrobot - IO - [MT Scale] Sent command: Z
2026-03-30 11:30:52,066 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 11:30:52,331 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 11:30:52,332 - pylabrobot - IO - [MT Scale] Received response: b'Z A\r\n'
2026-03-30 11:30:52,333 - pylabrobot - IO - [MT Scale] Sent command: SI
2026-03-30 11:30:52,335 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 11:30:52,364 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 11:30:52,367 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'


Frontend zero + read: 0.0 g


Place a container on the scale and press Enter... 


2026-03-30 11:30:54,533 - pylabrobot - IO - [MT Scale] Sent command: T
2026-03-30 11:30:54,534 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 11:30:54,904 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 11:30:54,905 - pylabrobot - IO - [MT Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 11:30:54,906 - pylabrobot - IO - [MT Scale] Sent command: TA
2026-03-30 11:30:54,907 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 11:30:55,000 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 11:30:55,001 - pylabrobot - IO - [MT Scale] Received response: b'TA A    0.00000 g\r\n'


Frontend tare weight: 0.0 g


AssertionError: 

---
## 6. Performance

In [24]:
import time

import numpy as np

# Stable read latency
times_stable = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout="stable")
  t1 = time.monotonic_ns()
  times_stable.append((t1 - t0) / 1e6)

# Immediate read latency
times_immediate = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout=0)
  t1 = time.monotonic_ns()
  times_immediate.append((t1 - t0) / 1e6)

print(f"Stable read:    {np.mean(times_stable):.2f} +/- {np.std(times_stable):.2f} ms")
print(f"Immediate read: {np.mean(times_immediate):.2f} +/- {np.std(times_immediate):.2f} ms")

2026-03-30 11:31:00,849 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 11:31:00,850 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 11:31:00,884 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 11:31:00,886 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 11:31:00,888 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 11:31:00,890 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 11:31:00,980 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 11:31:00,980 - pylabrobot - IO - [MT Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 11:31:00,981 - pylabrobot - IO - [MT Scale] Sent command: S
2026-03-30 11:31:00,985 - pylabrobot.io.serial - IO - [

Stable read:    93.20 +/- 19.10 ms
Immediate read: 47.39 +/- 7.81 ms


---
## 7. Teardown

In [25]:
plr_logger.info("=== Hardware validation ended ===")
await scale.stop()
print(f"Log file: {log_file}")

2026-03-30 11:31:07,091 - pylabrobot - INFO - === Hardware validation ended ===


Log file: ./logs/hardware_validation/2026-03-30_11-28-29_validation.log
